# D1 · From transaction evidence to paid basket events

Run this CPU notebook in order. It traces each original Excel row to a single cleaning decision, then combines paid merchandise lines into the purchase events shared by the four recommenders. The evidence workbook explains how to interpret cancellations, quantities, prices and product codes. Every audit step names the relevant sheet or the training-period code register. [Exception analysis notebook](../docs/technical/transaction_exception_analysis.ipynb) · [17-sheet evidence workbook](../results/transaction_exceptions.xlsx)

# D1 · 从异常证据到付费购物篮

使用 CPU，按顺序运行。先追踪每条原始 Excel 明细的清洗决定，再将有效商品行合并为四个推荐模型共用的购买事件。异常分析解释取消发票、数量、价格和商品编码的业务含义。每个审计步骤都标明证据所在的工作表或训练期编码登记表。[异常分析 notebook](../docs/technical/transaction_exception_analysis.ipynb) · [17 张异常分析表](../results/transaction_exceptions.xlsx)

## Locate the original workbook

In Colab, mount Drive and open the project’s `data/raw/Online Retail.xlsx`. When prompted, enter the path from the Files panel. Locally run from the repository root or its `notebooks/` directory. Results go under the chosen project folder.

## 定位原始工作簿

在 Colab 授权挂载 Drive，读取项目文件夹中的 `data/raw/Online Retail.xlsx`。如出现提示，从左侧文件面板复制完整路径。本地可在仓库根目录或 `notebooks/` 中运行。结果写入选定项目文件夹。

In [1]:
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    drive.mount("/content/drive")
    project = Path("/content/drive/MyDrive/CS5481_Project2_OnlineRetail")
    workbook = project / "data/raw/Online Retail.xlsx"
    if not workbook.is_file():
        workbook = Path(input("Workbook path / 工作簿完整路径：").strip())
        if not workbook.is_file():
            raise FileNotFoundError("Workbook not found / 找不到工作簿")
        if workbook.parent.name == "raw" and workbook.parent.parent.name == "data":
            project = workbook.parents[2]
        else:
            project = Path(input("Drive project folder for results / 保存结果的 Drive 项目文件夹完整路径：").strip())
            if not project.is_absolute() or not project.is_dir():
                raise ValueError("Select an existing project folder / 请选择已存在的项目文件夹")
else:
    project = Path.cwd()
    if project.name == "notebooks":
        project = project.parent
    workbook = project / "data/raw/Online Retail.xlsx"

if not workbook.is_file():
    raise FileNotFoundError(f"Workbook not found / 找不到工作簿：{workbook}")

output = project / "results/cleaning_audit.csv"
print("Workbook / 工作簿：", workbook)
print("Audit output / 审计表：", output)

Workbook / 工作簿： /tmp/retail-final-cleaning-BYTFneR0/project/data/raw/Online Retail.xlsx
Audit output / 审计表： /tmp/retail-final-cleaning-BYTFneR0/project/results/cleaning_audit.csv


## Verify the source and preserve the row numbers

Check the workbook checksum and eight source fields. Read invoice, product and customer identifiers as text and preserve their exact case. The first data record is Excel row 2. Compare the field counts with `missingness`. A missing description permits a product-coded purchase to survive. A missing country alone also leaves eligibility unchanged.

## 核验输入并保留行号

核对文件指纹与八个原始字段。按文本读取发票、商品与客户编号，保留原始大小写。第一条明细是 Excel 第 2 行。各字段的缺失数可与 `missingness` 对照。有效商品即使缺少描述仍可保留，展示时可用商品编码。仅缺少国家也不影响购买资格。

In [2]:
import hashlib
import pandas as pd
from IPython.display import display

WORKBOOK_SHA256 = "43465a06f2ccf7c8b5bd2892bc7defb52f97487934fe93b16ae4c3936424676d"
RAW_COLUMNS = [
    "InvoiceNo", "StockCode", "Description", "Quantity",
    "InvoiceDate", "UnitPrice", "CustomerID", "Country",
]


def read_workbook(path: Path) -> pd.DataFrame:
    with Path(path).open("rb") as stream:
        if hashlib.file_digest(stream, "sha256").hexdigest() != WORKBOOK_SHA256:
            raise ValueError("Workbook differs from the source identified in the research plan.")
        stream.seek(0)
        raw = pd.read_excel(stream, sheet_name="Online Retail", engine="openpyxl",
                            dtype={key: "string" for key in ("InvoiceNo", "StockCode", "CustomerID")},
                            keep_default_na=False, na_values=[""])
    if raw.columns.tolist() != RAW_COLUMNS:
        raise ValueError("Unexpected source schema.")
    raw.insert(0, "ExcelRow", range(2, len(raw) + 2))
    return raw
raw = read_workbook(workbook)
display(pd.DataFrame({"type": raw[RAW_COLUMNS].dtypes.astype(str), "missing_rows": raw[RAW_COLUMNS].isna().sum()}))
print(f"{len(raw):,} rows · {raw['InvoiceDate'].min()} — {raw['InvoiceDate'].max()}")

,type,missing_rows
InvoiceNo,string,0
StockCode,string,0
Description,object,1454
Quantity,int64,0
InvoiceDate,datetime64[ns],0
UnitPrice,float64,0
CustomerID,string,135080
Country,object,0


541,909 rows · 2010-12-01 08:26:00 — 2011-12-09 12:50:00


## Reproduce the five-rule baseline

The original audit is a checkpoint. Exactly identical lines are detected using all eight source fields. Raw condition counts can overlap. Only the first matching rule removes a given line. The five-step total leaves 392,692 candidate lines before checking merchandise codes.

## 复算五步基线

用全部八个原始字段识别完全重复行。原表条件可重叠，每条记录只由首次命中的规则移除一次。旧五步审计保留 392,692 条候选交易明细，这是进一步判断商品身份前的校验点。

In [3]:
def purchase_filter_flags(frame: pd.DataFrame) -> dict[str, pd.Series]:
    """Evaluate the five proposed purchase-sample rules in their audit order."""
    return {
        "exact_duplicate": frame.duplicated(subset=RAW_COLUMNS, keep="first"),
        "missing_customer_id": frame["CustomerID"].isna(),
        "cancelled_invoice": frame["InvoiceNo"].astype("string").str.startswith("C", na=False),
        "nonpositive_quantity": frame["Quantity"].le(0).fillna(False),
        "nonpositive_unit_price": frame["UnitPrice"].le(0).fillna(False),
    }


def audit_transactions(frame: pd.DataFrame) -> pd.DataFrame:
    """Count each flag on raw rows and assign each removed row to its first rule."""
    flags = purchase_filter_flags(frame)
    remaining = pd.Series(True, index=frame.index)
    records = []
    for step, (rule, flagged) in enumerate(flags.items(), start=1):
        before = int(remaining.sum())
        removed = int((remaining & flagged).sum())
        remaining &= ~flagged
        records.append({
            "step": step,
            "rule": rule,
            "raw_flagged": int(flagged.sum()),
            "rows_before": before,
            "rows_removed": removed,
            "rows_after": int(remaining.sum()),
        })
    return pd.DataFrame.from_records(records)

baseline = audit_transactions(raw)
display(baseline)

,step,rule,raw_flagged,rows_before,rows_removed,rows_after
0,1,exact_duplicate,5268,541909,5268,536641
1,2,missing_customer_id,135080,536641,135037,401604
2,3,cancelled_invoice,9288,401604,8872,392732
3,4,nonpositive_quantity,10624,392732,0,392732
4,5,nonpositive_unit_price,2517,392732,40,392692


## Read an individual cleaning step

`raw_flagged` counts matches across all source rows. `rows_before` counts the survivors entering this step. Only their first matches contribute to `rows_removed`. `rows_after` is the number carried forward. Follow an `ExcelRow` from the decision table to its `first_rule`, then read the reasons and evidence references for that rule in `cleaning_steps.csv`. The final retained row is a reconciliation total.

## 怎样阅读每一个清洗步骤

`raw_flagged` 是原表所有命中行数。`rows_before` 是进入该步的剩余行数，只有在这里首次命中的行才计入 `rows_removed`。`rows_after` 是传给下一步的行数。用 `ExcelRow` 在逐行决定表中找到 `first_rule`，再到 `cleaning_steps.csv` 查看对应理由及证据表。最后的保留行用于核对总数。

In [4]:
if raw["ExcelRow"].isna().any() or not raw["ExcelRow"].is_unique:
    raise ValueError("ExcelRow must uniquely identify each row of the source worksheet.")
frame = raw.reset_index(drop=True)
flags = purchase_filter_flags(frame)
surviving = pd.Series(True, index=frame.index)
first_rule = pd.Series("retained", index=frame.index, dtype="string")
records = []
def exclude(rule, flag, sheets, reason_en, reason_zh):
    mask = flag.fillna(False)
    removed = mask & surviving
    before = int(surviving.sum())
    count = int(removed.sum())
    first_rule.loc[removed] = rule
    surviving.loc[removed] = False
    records.append({
        "step": len(records) + 1, "rule": rule, "action": "exclude",
        "raw_flagged": int(mask.sum()), "rows_before": before,
        "rows_removed": count, "rows_after": before - count,
        "evidence_sheets": sheets, "reason_en": reason_en, "reason_zh": reason_zh,
    })

def show_steps(entries):
    with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
        display(pd.DataFrame(entries)[["step", "rule", "raw_flagged", "rows_before", "rows_removed",
                                       "rows_after", "evidence_sheets", "reason_en", "reason_zh"]])

## Duplicates and missing fields

Keep the first exact copy. The 135,080 lines without a customer ID span 3,710 invoices. In `customer_id_invoices`, none of these invoices supplies an ID on another line. Next check each required field separately, including the fields that are complete in this workbook. These checks preserve evidence for the identity, quantity, price and timing of a purchase.

## 重复与缺失：逐字段处理

完全相同的明细保留首条。缺客户编号的 135,080 行涉及 3,710 张发票。`customer_id_invoices` 显示，这些发票的其他行也都没有可补用的编号。随后逐项检查必要字段，原表中缺失数为零的字段也单列，用于核验每笔购买的身份、数量、价格与时间。

In [5]:
exclude("exact_duplicate", flags["exact_duplicate"], "filter_attribution",
        "Keep the first occurrence of an identical eight-field record.",
        "按八个原始字段识别完全相同的明细，保留首次出现的一行。")
exclude("missing_customer_id", flags["missing_customer_id"], "missingness, customer_id_invoices, filter_attribution",
        "An anonymous line cannot be assigned to a customer's history. The source audit retains these lines.",
        "匿名交易无法归入具体客户的历史，原始审计仍涵盖这些行。")
for rule, field, reason_en, reason_zh in [
    ("missing_invoice_no", "InvoiceNo", "An invoice identifier is required to construct a basket.", "缺少发票编号，无法构造购物篮。"),
    ("missing_stock_code", "StockCode", "An item identifier is required to assign a product.", "缺少商品编码，无法确定商品。"),
    ("missing_invoice_date", "InvoiceDate", "The timestamp determines the basket time and eligible history.", "缺少时间，无法确定购物篮发生时间与可用历史。"),
    ("missing_quantity", "Quantity", "The purchased quantity is unknown.", "缺少数量，无法核实购买件数。"),
    ("missing_unit_price", "UnitPrice", "The recorded unit price is unknown.", "缺少单价，无法核实计价金额。"),
]:
    exclude(rule, frame[field].isna(), "missingness", reason_en, reason_zh)
show_steps(records)

,step,rule,raw_flagged,rows_before,rows_removed,rows_after,evidence_sheets,reason_en,reason_zh
0,1,exact_duplicate,5268,541909,5268,536641,filter_attribution,Keep the first occurrence of an identical eight-field record.,按八个原始字段识别完全相同的明细，保留首次出现的一行。
1,2,missing_customer_id,135080,536641,135037,401604,"missingness, customer_id_invoices, filter_attribution",An anonymous line cannot be assigned to a customer's history. The source audit retains these lines.,匿名交易无法归入具体客户的历史，原始审计仍涵盖这些行。
2,3,missing_invoice_no,0,401604,0,401604,missingness,An invoice identifier is required to construct a basket.,缺少发票编号，无法构造购物篮。
3,4,missing_stock_code,0,401604,0,401604,missingness,An item identifier is required to assign a product.,缺少商品编码，无法确定商品。
4,5,missing_invoice_date,0,401604,0,401604,missingness,The timestamp determines the basket time and eligible history.,缺少时间，无法确定购物篮发生时间与可用历史。
5,6,missing_quantity,0,401604,0,401604,missingness,The purchased quantity is unknown.,缺少数量，无法核实购买件数。
6,7,missing_unit_price,0,401604,0,401604,missingness,The recorded unit price is unknown.,缺少单价，无法核实计价金额。


## Credits, negative quantities, zero quantities and negative prices

All 9,288 C-prefixed lines have negative quantities. Some concern discounts. Among 1,336 non-C negative lines, all lack a customer ID and have zero unit price. No earlier positive line shares their invoice number. Keep those distinctions in the audit. The two negative-price lines explicitly say `Adjust bad debt`. A credit posted later leaves a customer's earlier paid purchase in its historical basket.

## 取消发票、负数量、零数量与负价

9,288 条 C 前缀明细的数量均为负，其中有折扣记录。另有 1,336 条非 C 负数量行，全部缺客户编号且单价为零，查不到同一发票号下更早的正数量明细。各类分别审计。两条负价明细原文为 `Adjust bad debt`（坏账调整）。客户先前正常计价的购买保留在当时的购物篮中，后续冲销单独追溯。

In [6]:
start = len(records)
exclude("cancelled_invoice", flags["cancelled_invoice"], "C_invoices, history_summary, case_examples",
        "A C-prefixed document contributes no new purchase. Earlier positive lines remain in history.",
        "C 前缀发票不产生新的购买事件，较早的正向购买仍保留在历史中。")
exclude("negative_quantity", frame["Quantity"].lt(0), "Q_nonpositive, nonC_negative, note_summary",
        "Negative units do not add purchased items. Notes can describe damage or adjustments without identifying a buyer.",
        "负数量不形成新增购买。备注可能记录损坏或调整，但不足以确认买家。")
exclude("zero_quantity", frame["Quantity"].eq(0), "Q_nonpositive, filter_attribution",
        "Zero units add no purchase event.", "零数量不产生购买事件。")
exclude("negative_unit_price", frame["UnitPrice"].lt(0), "P_nonpositive, case_examples, note_summary",
        "A negative unit price is outside the paid-item definition. Both observed examples record bad-debt adjustments.",
        "负单价不符合计价商品定义，原表两条实例的备注均为坏账调整。")
show_steps(records[start:])

,step,rule,raw_flagged,rows_before,rows_removed,rows_after,evidence_sheets,reason_en,reason_zh
0,8,cancelled_invoice,9288,401604,8872,392732,"C_invoices, history_summary, case_examples",A C-prefixed document contributes no new purchase. Earlier positive lines remain in history.,C 前缀发票不产生新的购买事件，较早的正向购买仍保留在历史中。
1,9,negative_quantity,10624,392732,0,392732,"Q_nonpositive, nonC_negative, note_summary",Negative units do not add purchased items. Notes can describe damage or adjustments without identifying a buyer.,负数量不形成新增购买。备注可能记录损坏或调整，但不足以确认买家。
2,10,zero_quantity,0,392732,0,392732,"Q_nonpositive, filter_attribution",Zero units add no purchase event.,零数量不产生购买事件。
3,11,negative_unit_price,2,392732,0,392732,"P_nonpositive, case_examples, note_summary",A negative unit price is outside the paid-item definition. Both observed examples record bad-debt adjustments.,负单价不符合计价商品定义，原表两条实例的备注均为坏账调整。


## Zero prices and possible complimentary items

Partition every zero-price line using invoice evidence. The price step reaches 40 identified positive-quantity rows. Six are Manual. Five product lines have the same SKU priced on the invoice, 23 have another priced line, and six product lines have no priced companion. The 23 same-SKU product matches reported in the exception notebook across the whole workbook comprise five identified and 18 anonymous rows. These are two different denominators. The recorded prices do not establish whether an item was a gift. Preserve each qualifying priced line on the invoice.

## 零价与疑似随单赠送

对原表全部零价行检查同票上下文。运行到单价步骤时仍有 40 行，均有客户编号且数量为正。其中 6 行为 Manual；5 条商品明细同票同商品另有正价行，23 条同票另有其他计价行，6 条商品明细同票无正价行。异常分析在全表找出的 23 条「同票同商品还有正价」记录，则是 5 条有编号与 18 条无编号明细之和，两处的 23 采用不同口径。零价原因未注明，只排除零价行，同票中符合条件的计价购买仍保留。

In [7]:
def _zero_price_contexts(raw: pd.DataFrame, cancelled: pd.Series) -> dict[str, pd.Series]:
    """Partition all raw zero-price lines, including lines removed by earlier rules.

    Companions have positive quantity and unit price on a non-C invoice with a
    known invoice number. A same-item match additionally requires the exact
    StockCode and CustomerID, with a missing customer key matching only another
    missing key. A priced line can itself be a fee. Search the complete invoice.
    These comparisons describe a zero price. All four cases have the same
    purchase-label decision, irrespective of where the query cutoff falls.
    """
    zero = raw["UnitPrice"].eq(0).fillna(False)
    manual = raw["StockCode"].isin(["M", "m"])
    keys = ["InvoiceNo", "CustomerID", "StockCode"]
    priced = raw.loc[
        ~cancelled & raw["Quantity"].gt(0) & raw["UnitPrice"].gt(0), keys
    ].dropna(subset=["InvoiceNo"]).drop_duplicates()
    priced_keys = priced.dropna(subset=["StockCode"])
    # Limit the three-key comparison to the small set of zero-price rows.
    same_item = pd.Series(False, index=raw.index)
    same_item.loc[zero] = pd.MultiIndex.from_frame(raw.loc[zero, keys]).isin(
        pd.MultiIndex.from_frame(priced_keys)
    )
    same_invoice = raw["InvoiceNo"].isin(priced["InvoiceNo"])
    return {
        "zero_price_manual": zero & manual,
        "zero_price_same_item": zero & ~manual & same_item,
        "zero_price_other_item": zero & ~manual & ~same_item & same_invoice,
        "zero_price_no_priced_companion": zero & ~manual & ~same_invoice,
    }

start = len(records)
zero_price = _zero_price_contexts(frame, flags["cancelled_invoice"])
zero_sheets = "P_nonpositive, case_examples, history_summary"
exclude("zero_price_manual", zero_price["zero_price_manual"], "P_nonpositive, note_summary",
        "An exact manual-entry code (M or m) carries a recorded zero unit price.",
        "手工记录编码 M 或 m 的记录单价为零。")
exclude("zero_price_same_item", zero_price["zero_price_same_item"], zero_sheets,
        "A priced line shares the invoice, customer key and SKU. The zero-price row adds no paid units.",
        "同票另有计价行匹配该客户键和商品编码，零价行不增加计价购买件数。")
exclude("zero_price_other_item", zero_price["zero_price_other_item"], zero_sheets,
        "This invoice has a priced line, but none matches the customer key and SKU. A priced line can also represent a fee.",
        "同票有计价行，但没有计价行同时匹配本行的客户键和商品编码。计价行也可能是费用。")
exclude("zero_price_no_priced_companion", zero_price["zero_price_no_priced_companion"], zero_sheets,
        "No non-C line with positive quantity and price was found under this exact invoice number.",
        "该发票编号下未查到非 C 且数量和单价均为正的计价行。")
show_steps(records[start:])
assert records[-1]["rows_after"] == baseline.iloc[-1]["rows_after"]

,step,rule,raw_flagged,rows_before,rows_removed,rows_after,evidence_sheets,reason_en,reason_zh
0,12,zero_price_manual,6,392732,6,392726,"P_nonpositive, note_summary",An exact manual-entry code (M or m) carries a recorded zero unit price.,手工记录编码 M 或 m 的记录单价为零。
1,13,zero_price_same_item,23,392726,5,392721,"P_nonpositive, case_examples, history_summary","A priced line shares the invoice, customer key and SKU. The zero-price row adds no paid units.",同票另有计价行匹配该客户键和商品编码，零价行不增加计价购买件数。
2,14,zero_price_other_item,372,392721,23,392698,"P_nonpositive, case_examples, history_summary","This invoice has a priced line, but none matches the customer key and SKU. A priced line can also represent a fee.",同票有计价行，但没有计价行同时匹配本行的客户键和商品编码。计价行也可能是费用。
3,15,zero_price_no_priced_companion,2114,392698,6,392692,"P_nonpositive, case_examples, history_summary",No non-C line with positive quantity and price was found under this exact invoice number.,该发票编号下未查到非 C 且数量和单价均为正的计价行。


## Freeze item-code decisions using training evidence

Review codes against literal descriptions dated from 1 December 2010 through 30 September 2011. The register gives the original Excel row and counts before and after the basic filters in that period. Verify these counts below. Use exact, case-sensitive codes. For example, 23444 means next-day carriage, while 23099 is a carriage lantern. PADS describes cushion pads. Ambiguous cases stay eligible unless there is evidence for excluding the whole code.

## 用训练期证据冻结商品编码规则

回查 2010 年 12 月 1 日至 2011 年 9 月 30 日的描述原文。登记表列出原始行号，以及该期间基础过滤前后的记录数，下方重新核对这些数值。使用区分大小写的完整编码。例如，23444 记录次日配送费，23099 是商品名称中带有 carriage 的灯，PADS 对应靠垫。对于依据不足的编码，继续按一般购买条件判断。

In [8]:
_POLICY = {
    "delivery": (
        "exclude", "Postage or carriage pays for delivery.",
        "邮费或运费支付的是配送服务。",
    ),
    "packing_fee": (
        "exclude", "The line bills a packing fee.",
        "该行收取包装费。",
    ),
    "financial_fee": (
        "exclude", "The description identifies a payment, platform or commission fee.",
        "描述表明该行记录银行、平台费用或佣金。",
    ),
    "bad_debt": (
        "exclude", "The line records a bad-debt adjustment.",
        "该行属于坏账调整。",
    ),
    "discount": (
        "exclude", "The line records a discount on a transaction.",
        "该行记录交易折扣。",
    ),
    "manual": (
        "exclude", "The source description only says Manual. It identifies no particular product to rank.",
        "原文描述只有 Manual，无法据此识别可供推荐的具体商品。",
    ),
    "sample": (
        "exclude", "The generic sample code does not identify a particular item to recommend.",
        "样品通用编码未指明可推荐的具体商品。",
    ),
    "voucher": (
        "exclude", "The denomination identifies prepaid store credit. This project ranks merchandise.",
        "面额对应店铺预付购物额度；本项目的推荐对象是商品。",
    ),
    "product": (
        "retain", "The training description names a physical product.",
        "训练期描述指明了具体实物商品。",
    ),
    "unresolved": (
        "retain", "Training rows contain only a missing description or a channel name. "
                  "Evidence is insufficient for a code-wide exclusion.",
        "训练期描述缺失或只记录销售渠道，没有足够证据对整个商品编码设定排除规则。",
    ),
}

# Columns: exact code, policy, literal example description, Excel row, time,
# all raw training rows, and training rows surviving the basic purchase filters.
_TRAINING_EVIDENCE = (
    ("23444", "delivery", "Next Day Carriage", 205750, "2011-05-26 16:11", 38, 36),
    ("23574", "packing_fee", "PACKING CHARGE", 252607, "2011-07-06 13:04", 9, 7),
    ("AMAZONFEE", "financial_fee", "AMAZON FEE", 14516, "2010-12-07 12:41", 27, 0),
    ("B", "bad_debt", "Adjust bad debt", 299984, "2011-08-12 14:50", 3, 0),
    ("BANK CHARGES", "financial_fee", "Bank Charges", 4408, "2010-12-02 15:08", 27, 7),
    ("C2", "delivery", "CARRIAGE", 1425, "2010-12-01 14:05", 102, 94),
    ("CRUK", "financial_fee", "CRUK Commission", 317510, "2011-08-30 10:49", 6, 0),
    ("D", "discount", "Discount", 143, "2010-12-01 09:41", 61, 0),
    ("DOT", "delivery", "DOTCOM POSTAGE", 1816, "2010-12-01 14:32", 603, 6),
    ("M", "manual", "Manual", 2241, "2010-12-01 15:35", 411, 188),
    ("m", "manual", "Manual", 40385, "2010-12-21 15:18", 1, 0),
    ("POST", "delivery", "POSTAGE", 47, "2010-12-01 08:45", 872, 770),
    ("S", "sample", "SAMPLES", 14438, "2010-12-07 12:03", 55, 0),
    ("gift_0001_10", "voucher", "Dotcomgiftshop Gift Voucher £10.00", 112444, "2011-03-07 17:14", 8, 0),
    ("gift_0001_20", "voucher", "Dotcomgiftshop Gift Voucher £20.00", 44796, "2011-01-05 14:44", 9, 0),
    ("gift_0001_30", "voucher", "Dotcomgiftshop Gift Voucher £30.00", 44727, "2011-01-05 14:44", 8, 0),
    ("gift_0001_40", "voucher", "Dotcomgiftshop Gift Voucher £40.00", 38250, "2010-12-20 10:14", 3, 0),
    ("gift_0001_50", "voucher", "Dotcomgiftshop Gift Voucher £50.00", 42059, "2010-12-23 13:26", 4, 0),
    ("21518", "product", "BANK ACCOUNT  GREETING  CARD ", 3343, "2010-12-02 10:39", 39, 17),
    ("23099", "product", "FRENCH CARRIAGE LANTERN", 191482, "2011-05-16 14:30", 42, 39),
    ("85123A", "product", "WHITE HANGING HEART T-LIGHT HOLDER", 2, "2010-12-01 08:26", 1813, 1581),
    ("PADS", "product", "PADS TO MATCH ALL CUSHIONS", 157197, "2011-04-15 09:27", 4, 3),
    ("DCGS0003", "product", "BOXED GLASS ASHTRAY", 24908, "2010-12-10 14:59", 5, 0),
    ("DCGS0004", "product", "HAYNES CAMPER SHOULDER BAG", 170785, "2011-04-27 17:23", 1, 0),
    ("DCGS0055", "unresolved", None, 74827, "2011-01-28 13:08", 1, 0),
    ("DCGS0057", "unresolved", None, 75297, "2011-01-28 15:01", 1, 0),
    ("DCGS0066P", "unresolved", None, 279259, "2011-07-26 11:52", 1, 0),
    ("DCGS0067", "unresolved", "ebay", 279258, "2011-07-26 11:52", 1, 0),
    ("DCGS0068", "unresolved", "ebay", 279257, "2011-07-26 11:52", 1, 0),
    ("DCGS0069", "product", "OOH LA LA DOGS COLLAR", 75055, "2011-01-28 14:25", 2, 0),
    ("DCGS0070", "product", "CAMOUFLAGE DOG COLLAR", 40054, "2010-12-21 13:06", 2, 0),
    ("DCGS0071", "unresolved", None, 279254, "2011-07-26 11:51", 1, 0),
    ("DCGS0072", "unresolved", None, 74840, "2011-01-28 13:08", 1, 0),
    ("DCGS0073", "unresolved", "ebay", 279253, "2011-07-26 11:51", 1, 0),
    ("DCGS0074", "unresolved", None, 74841, "2011-01-28 13:09", 1, 0),
    ("DCGS0076", "product", "SUNJAR LED NIGHT NIGHT LIGHT", 21328, "2010-12-09 14:09", 2, 0),
    ("DCGSSBOY", "product", "BOYS PARTY BAG", 84018, "2011-02-07 14:04", 10, 0),
    ("DCGSSGIRL", "product", "GIRLS PARTY BAG", 84019, "2011-02-07 14:04", 11, 0),
)

def merchandise_rules() -> pd.DataFrame:
    """Return the frozen exclusion decisions and reviewed retained codes with evidence."""
    return pd.DataFrame.from_records(
        [
            (code, *_POLICY[policy], *evidence)
            for code, policy, *evidence in _TRAINING_EVIDENCE
        ],
        columns=[
            "StockCode", "action", "reason_en", "reason_zh", "training_description",
            "training_example_excel_row", "training_example_date", "training_rows",
            "training_basic_rows",
        ],
    )

In [9]:
registry = merchandise_rules()
train_start = pd.Timestamp("2010-12-01")
cutoff = pd.Timestamp("2011-10-01")
training = raw.loc[raw["InvoiceDate"].ge(train_start) & raw["InvoiceDate"].lt(cutoff)]
basic = raw.loc[~pd.DataFrame(flags).any(axis=1)]
basic_training = basic.loc[basic["InvoiceDate"].ge(train_start) & basic["InvoiceDate"].lt(cutoff)]
evidence = raw.set_index("ExcelRow").loc[registry["training_example_excel_row"]]
assert evidence["StockCode"].tolist() == registry["StockCode"].tolist()
assert evidence["InvoiceDate"].tolist() == pd.to_datetime(registry["training_example_date"]).tolist()
assert evidence["Description"].fillna("").tolist() == registry["training_description"].fillna("").tolist()
for lines, field in [(training, "training_rows"), (basic_training, "training_basic_rows")]:
    counts = lines.groupby("StockCode").size()
    assert registry["StockCode"].map(counts).fillna(0).tolist() == registry[field].tolist()
with pd.option_context("display.max_rows", None):
    display(registry[["StockCode", "action", "training_description", "training_example_excel_row",
                      "training_rows", "training_basic_rows"]])

,StockCode,action,training_description,training_example_excel_row,training_rows,training_basic_rows
0,23444,exclude,Next Day Carriage,205750,38,36
1,23574,exclude,PACKING CHARGE,252607,9,7
2,AMAZONFEE,exclude,AMAZON FEE,14516,27,0
3,B,exclude,Adjust bad debt,299984,3,0
4,BANK CHARGES,exclude,Bank Charges,4408,27,7
5,C2,exclude,CARRIAGE,1425,102,94
6,CRUK,exclude,CRUK Commission,317510,6,0
7,D,exclude,Discount,143,61,0
8,DOT,exclude,DOTCOM POSTAGE,1816,603,6
9,M,exclude,Manual,2241,411,188


## Apply one decision per excluded code

Across the full workbook 571 uppercase M lines and one lowercase m line carry only the description Manual. The underlying item is unidentified, including both lines of invoice 561916. B has three entries explicitly labelled Adjust bad debt, including one positive-price entry. Keep every code’s raw count, even when earlier rules removed all its lines. The code register supplies the reason and a source row for each exclusion.

## 每个排除编码单设一步

全表有 571 条大写 M 和 1 条小写 m，描述均只有 Manual。发票 561916 的两行也只能看到这个标签，具体商品或费用无法确定。B 则有三条明确写着 Adjust bad debt 的账务记录，其中一条单价为正。即使较早的步骤已排除某个编码的全部记录，仍列出该编码在原表的命中数，并在登记表中保留排除理由及证据行号。

In [10]:
start = len(records)
frozen = merchandise_rules()
code_sheets = {
    "23444": "C_invoices, P_nonpositive", "23574": "C_invoices, P_nonpositive",
    "AMAZONFEE": "C_invoices, note_summary", "B": "P_nonpositive, case_examples, note_summary",
    "BANK CHARGES": "C_invoices, note_summary", "C2": "C_invoices, P_nonpositive, note_summary",
    "CRUK": "C_invoices, note_summary", "D": "C_invoices, case_examples, note_summary",
    "DOT": "C_invoices, P_nonpositive, note_summary", "M": "C_invoices, P_nonpositive, note_summary",
    "POST": "C_invoices, P_nonpositive, note_summary", "S": "C_invoices, note_summary",
    "gift_0001_10": "P_nonpositive", "gift_0001_20": "P_nonpositive", "gift_0001_30": "P_nonpositive",
}
for item in frozen.loc[frozen["action"].eq("exclude")].itertuples(index=False):
    sources = ["merchandise_rules.csv"]
    if item.StockCode in code_sheets:
        sources.append(code_sheets[item.StockCode])
    exclude(f"non_merchandise_{item.StockCode}", frame["StockCode"].eq(item.StockCode),
            ", ".join(sources), item.reason_en, item.reason_zh)
show_steps(records[start:])

,step,rule,raw_flagged,rows_before,rows_removed,rows_after,evidence_sheets,reason_en,reason_zh
0,16,non_merchandise_23444,81,392692,79,392613,"merchandise_rules.csv, C_invoices, P_nonpositive",Postage or carriage pays for delivery.,邮费或运费支付的是配送服务。
1,17,non_merchandise_23574,17,392613,14,392599,"merchandise_rules.csv, C_invoices, P_nonpositive",The line bills a packing fee.,该行收取包装费。
2,18,non_merchandise_AMAZONFEE,34,392599,0,392599,"merchandise_rules.csv, C_invoices, note_summary","The description identifies a payment, platform or commission fee.",描述表明该行记录银行、平台费用或佣金。
3,19,non_merchandise_B,3,392599,0,392599,"merchandise_rules.csv, P_nonpositive, case_examples, note_summary",The line records a bad-debt adjustment.,该行属于坏账调整。
4,20,non_merchandise_BANK CHARGES,37,392599,12,392587,"merchandise_rules.csv, C_invoices, note_summary","The description identifies a payment, platform or commission fee.",描述表明该行记录银行、平台费用或佣金。
5,21,non_merchandise_C2,144,392587,133,392454,"merchandise_rules.csv, C_invoices, P_nonpositive, note_summary",Postage or carriage pays for delivery.,邮费或运费支付的是配送服务。
6,22,non_merchandise_CRUK,16,392454,0,392454,"merchandise_rules.csv, C_invoices, note_summary","The description identifies a payment, platform or commission fee.",描述表明该行记录银行、平台费用或佣金。
7,23,non_merchandise_D,77,392454,0,392454,"merchandise_rules.csv, C_invoices, case_examples, note_summary",The line records a discount on a transaction.,该行记录交易折扣。
8,24,non_merchandise_DOT,710,392454,16,392438,"merchandise_rules.csv, C_invoices, P_nonpositive, note_summary",Postage or carriage pays for delivery.,邮费或运费支付的是配送服务。
9,25,non_merchandise_M,571,392438,279,392159,"merchandise_rules.csv, C_invoices, P_nonpositive, note_summary",The source description only says Manual. It identifies no particular product to rank.,原文描述只有 Manual，无法据此识别可供推荐的具体商品。


## Reconcile row decisions and calculate line amounts

An excluded row has one first rule. Check that 391,060 retained paid lines and all exclusions add back to 541,909 original rows. Compute quantity times the recorded unit price for each retained line. Keep source precision, including the three PADS purchases charged at £0.001 per unit.

## 核对逐行去向并计算行金额

每条被排除记录都有唯一的首条命中规则。将 391,060 条保留行与所有净排除行加总，核对是否等于原表的 541,909 行。对保留行先按原始单价乘以数量计算金额。三条 PADS 明细的单价为 £0.001，计算中保留原精度。

In [11]:
retained = int(surviving.sum())
records.append({
    "step": len(records) + 1, "rule": "retained", "action": "retain",
    "raw_flagged": retained, "rows_before": retained, "rows_removed": 0,
    "rows_after": retained, "evidence_sheets": "filter_attribution, sku_names, merchandise_rules.csv",
    "reason_en": "Keep identified, dated, positive-quantity, positive-price merchandise lines. Preserve exact SKU and invoice IDs, including suffixes.",
    "reason_zh": "保留客户、时间、发票与商品明确且数量和单价均为正的商品明细，按原值保留发票及商品编码（含后缀）。",
})
decisions = pd.DataFrame({"ExcelRow": frame["ExcelRow"], "first_rule": first_rule})
purchases = frame.loc[surviving].copy()
purchases["line_amount_gbp"] = purchases["Quantity"] * purchases["UnitPrice"]
steps = pd.DataFrame.from_records(records)
assert len(decisions) == len(raw) == len(purchases) + steps["rows_removed"].sum()
removed = steps.loc[steps["action"].eq("exclude")].set_index("rule")
assert decisions["first_rule"].value_counts().reindex(removed.index, fill_value=0).eq(removed["rows_removed"]).all()
assert decisions.loc[decisions["first_rule"].eq("retained"), "ExcelRow"].tolist() == purchases["ExcelRow"].tolist()
display(steps[["step", "rule", "raw_flagged", "rows_removed", "rows_after"]])

,step,rule,raw_flagged,rows_removed,rows_after
0,1,exact_duplicate,5268,5268,536641
1,2,missing_customer_id,135080,135037,401604
2,3,missing_invoice_no,0,0,401604
3,4,missing_stock_code,0,0,401604
4,5,missing_invoice_date,0,0,401604
5,6,missing_quantity,0,0,401604
6,7,missing_unit_price,0,0,401604
7,8,cancelled_invoice,9288,8872,392732
8,9,negative_quantity,10624,0,392732
9,10,zero_quantity,0,0,392732


## Which rule actually removed each exception?

Count the first decision for every line in selected evidence groups. In particular, the non-C negative quantities disappear at the customer-ID step, all three bad-debt entries lack a customer ID, and the 572 Manual rows reach several different steps. The 23 same-SKU zero-price product lines split between anonymous and identified buyers. One source row can appear in multiple evidence groups. Within each group, the columns add up to its raw row count.

## 异常表中的行最终在哪一步被处理？

按原始行号追踪各类异常第一次命中的规则。特别核对非 C 负数量、三条坏账调整、572 条 Manual 以及 23 条同票同商品的零价商品行。一个原始行可以进入多个证据组；在每个组内，各规则的行数之和必须等于该组的原始行数。只在组内加总。

In [12]:
evidence_groups = {
    "C_invoices · C prefix": flags["cancelled_invoice"],
    "nonC_negative · negative quantity": ~flags["cancelled_invoice"] & raw["Quantity"].lt(0),
    "P_nonpositive · price <= 0": raw["UnitPrice"].le(0),
    "missingness · no description": raw["Description"].isna(),
    "P_nonpositive / note_summary · M or m": raw["StockCode"].isin(["M", "m"]),
    "case_examples · B / Adjust bad debt": raw["StockCode"].eq("B"),
    "P_nonpositive · same invoice & SKU mixed prices / products": zero_price["zero_price_same_item"],
}
reason_counts = {}
for label, mask in evidence_groups.items():
    counts = decisions.loc[mask, "first_rule"].value_counts()
    assert counts.sum() == mask.sum()
    reason_counts[label] = counts
crosswalk = pd.DataFrame(reason_counts).fillna(0).astype(int).T
crosswalk.insert(0, "raw_rows", crosswalk.sum(axis=1))
display(crosswalk)

first_rule,raw_rows,cancelled_invoice,exact_duplicate,missing_customer_id,non_merchandise_M,zero_price_manual,zero_price_no_priced_companion,zero_price_other_item,zero_price_same_item
C_invoices · C prefix,9288,8872,37,379,0,0,0,0,0
nonC_negative · negative quantity,1336,0,0,1336,0,0,0,0,0
P_nonpositive · price <= 0,2517,0,5,2472,0,6,6,23,5
missingness · no description,1454,0,0,1454,0,0,0,0,0
P_nonpositive / note_summary · M or m,572,175,5,107,279,6,0,0,0
case_examples · B / Adjust bad debt,3,0,0,3,0,0,0,0,0
P_nonpositive · same invoice & SKU mixed prices / products,23,0,0,18,0,0,0,0,5


## Use one event per customer, invoice and product

Sum quantities and line amounts over the exact customer, invoice and product key. Use the latest retained timestamp on the entire customer invoice for every event in that basket. This keeps all items together when a basket crosses a time boundary. Use basket completion times when forming time windows for later training or evaluation.

## 按客户、发票、商品建立唯一购买事件

按客户、发票、商品三个字段合并数量与行金额。每个购物篮使用该客户这张发票全部有效行中的最晚时间，篮内各商品共享此时间。这样即使某次交易跨过时间边界，整篮也会归入同一时间窗。后续构建训练或评估样本时依据购物篮完成时间截取记录。

In [13]:
INVOICE_KEYS = ["CustomerID", "InvoiceNo"]
ITEM_KEYS = [*INVOICE_KEYS, "StockCode"]


def build_basket_events(purchases: pd.DataFrame) -> pd.DataFrame:
    """Aggregate each item within a customer invoice at the basket's completion time."""
    # Use the original row order for reproducible sums, including fractional pennies.
    ordered = purchases.sort_values("ExcelRow")
    events = ordered.groupby(ITEM_KEYS, as_index=False, sort=False).agg(
        basket_time=("InvoiceDate", "max"),
        quantity=("Quantity", "sum"),
        amount_gbp=("line_amount_gbp", "sum"),
        source_lines=("ExcelRow", "size"),
    )
    # The latest retained timestamp across all items closes the whole invoice.
    events["basket_time"] = events.groupby(INVOICE_KEYS, sort=False)["basket_time"].transform("max")
    columns = ["CustomerID", "InvoiceNo", "basket_time", "StockCode", "quantity", "amount_gbp", "source_lines"]
    return events.sort_values(["CustomerID", "basket_time", "InvoiceNo", "StockCode"], ignore_index=True)[columns]


def training_basket_counts(purchases: pd.DataFrame, cutoff: pd.Timestamp) -> dict[str, int]:
    """Count multi-time invoices and multi-price items in baskets completed before cutoff."""
    completion = purchases.groupby(INVOICE_KEYS, sort=False)["InvoiceDate"].transform("max")
    history = purchases.loc[completion.lt(cutoff)]
    invoice_times = history.groupby(INVOICE_KEYS, sort=False)["InvoiceDate"].nunique()
    item_prices = history.groupby(ITEM_KEYS, sort=False)["UnitPrice"].nunique()
    return {
        "multi_time_invoices": int(invoice_times.gt(1).sum()),
        "multi_price_invoice_items": int(item_prices.gt(1).sum()),
    }
import math

events = build_basket_events(purchases)
assert not events.duplicated(ITEM_KEYS).any()
assert events["source_lines"].sum() == len(purchases)
assert events["quantity"].sum() == purchases["Quantity"].sum()
assert math.isclose(events["amount_gbp"].sum(), purchases["line_amount_gbp"].sum(), rel_tol=1e-12)
display(pd.DataFrame({"five_rule_baseline": training_basket_counts(basic, cutoff),
                      "paid_merchandise": training_basket_counts(purchases, cutoff)}))

,five_rule_baseline,paid_merchandise
multi_time_invoices,27,27
multi_price_invoice_items,47,32


## Trace four cases through the decisions

On 550188, eight priced units of SKU 22636 yield one event worth £61.20. Its zero-price unit remains traceable as a separate source line. On 539318, two prices for SKU 21213 give 24 × £0.55 + 120 × £0.42 = £63.60 for 144 units. On 541431 the system records a large paid purchase. A later credit C541433 is traceable separately. Invoice 561916 illustrates a different decision for each of its two Manual rows.

## 从原始发票核查每一步的实际影响

发票 550188 的商品 22636 保留 8 件计价商品，形成 £61.20 的一次购买事件，零价的 1 件仍可通过原始行号追溯。发票 539318 的商品 21213 有两种单价，24 × £0.55 + 120 × £0.42 = £63.60，共 144 件。发票 541431 的大额购买保留，随后记录的贷项 C541433 单独追溯。发票 561916 的两条 Manual 明细则分别由不同规则排除。

In [14]:
examples = [("550188", "22636"), ("539318", "21213"), ("541431", "23166"),
            ("C541433", "23166"), ("561916", "M")]
selected = pd.MultiIndex.from_frame(raw[["InvoiceNo", "StockCode"]]).isin(examples)
case_rows = raw.loc[selected].merge(decisions, on="ExcelRow", validate="one_to_one")
display(case_rows[["ExcelRow", "CustomerID", "InvoiceNo", "InvoiceDate", "StockCode",
                   "Description", "Quantity", "UnitPrice", "first_rule"]])
display(events.loc[pd.MultiIndex.from_frame(events[["InvoiceNo", "StockCode"]]).isin(examples)])

,ExcelRow,CustomerID,InvoiceNo,InvoiceDate,StockCode,Description,Quantity,UnitPrice,first_rule
0,34089,12348,539318,2010-12-16 19:09:00,21213,PACK OF 72 SKULL CAKE CASES,24,0.55,retained
1,34090,12348,539318,2010-12-16 19:09:00,21213,PACK OF 72 SKULL CAKE CASES,120,0.42,retained
2,61621,12346,541431,2011-01-18 10:01:00,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,retained
3,61626,12346,C541433,2011-01-18 10:17:00,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,1.04,cancelled_invoice
4,157031,12457,550188,2011-04-14 18:57:00,22636,CHILDS BREAKFAST SET CIRCUS PARADE,8,7.65,retained
5,157044,12457,550188,2011-04-14 18:57:00,22636,CHILDS BREAKFAST SET CIRCUS PARADE,1,0.00,zero_price_same_item
6,285659,15581,561916,2011-08-01 11:44:00,M,Manual,1,0.00,zero_price_manual
7,285660,15581,561916,2011-08-01 11:44:00,M,Manual,1,250.00,non_merchandise_M


,CustomerID,InvoiceNo,basket_time,StockCode,quantity,amount_gbp,source_lines
0,12346,541431,2011-01-18 10:01:00,23166,74215,77183.6,1
184,12348,539318,2010-12-16 19:09:00,21213,144,63.6,2
7665,12457,550188,2011-04-14 18:57:00,22636,8,61.2,1


## Export and locate the four-model purchase dataset

Use `data/processed/basket_events.csv.gz` as the common purchase-event dataset. The two companion tables hold the retained source lines and the first decision for each original Excel row. The manifest records row counts and file hashes. The full-workbook audit includes the incomplete month of December 2011. Model histories and targets must follow the research plan’s monthly cutoffs. [Research plan](../docs/CS5481_Project2_Research_Development_Plan_3.4.en.md) · [Chen (2015), UCI Online Retail, CC BY 4.0](https://doi.org/10.24432/C5BW33)

## 导出并定位四个模型共用的数据集

四个模型共用 `data/processed/basket_events.csv.gz`。配套明细表保存有效购买的原始行，逐行决定表保存每个原始行号首次命中的规则。清单记录各文件行数和校验值。全表清洗包含不完整的 2011 年 12 月，后续训练历史与预测目标按研究计划的月度截止点取数。[研究计划](../docs/CS5481_Project2_Research_Development_Plan_3.4.md) · [Chen（2015），UCI Online Retail，CC BY 4.0](https://doi.org/10.24432/C5BW33)

In [15]:
import gzip
import json
import os
from contextlib import ExitStack
from tempfile import TemporaryDirectory

def cleaning_manifest(raw, baseline, steps, purchases, events):
    return {
        "source": {
            "dataset_doi": "10.24432/C5BW33",
            "workbook_sha256": WORKBOOK_SHA256,
            "sheet": "Online Retail",
            "raw_fields": RAW_COLUMNS,
            "excel_first_data_row": 2,
            "rows": len(raw),
            "first_timestamp": raw["InvoiceDate"].min().isoformat(),
            "last_timestamp": raw["InvoiceDate"].max().isoformat(),
        },
        "merchandise_evidence_period": {"start": "2010-12-01", "end_exclusive": "2011-10-01"},
        "counts": {
            "after_five_rules": int(baseline.iloc[-1]["rows_after"]),
            "removed_total": int(steps["rows_removed"].sum()),
            "paid_merchandise_lines": len(purchases),
            "basket_item_events": len(events),
            "baskets": len(events[["CustomerID", "InvoiceNo"]].drop_duplicates()),
            "customers": purchases["CustomerID"].nunique(),
            "stock_codes": purchases["StockCode"].nunique(),
            "paid_units": int(purchases["Quantity"].sum()),
            "recorded_amount_gbp": float(format(purchases["line_amount_gbp"].sum(), ".15g")),
        },
        "event_key": ["CustomerID", "InvoiceNo", "StockCode"],
        "basket_time": "Latest retained line timestamp on the customer invoice.",
        "scope": "Whole-workbook cleaning. Slice by basket_time before constructing model histories or labels.",
    }


def save_cleaning_results(project, baseline_path, baseline, steps, decisions, purchases, events, rules, manifest):
    """Stage each artifact beside its destination, then publish the checksum manifest last."""
    results = project / "results"
    processed = project / "data/processed"
    frames = [
        (baseline_path, baseline),
        (results / "cleaning_steps.csv", steps),
        (results / "merchandise_rules.csv", rules),
        (processed / "row_decisions.csv.gz", decisions),
        (processed / "purchase_lines.csv.gz", purchases),
        (processed / "basket_events.csv.gz", events),
    ]
    manifest_path = results / "data_manifest.json"
    if baseline_path.suffix.lower() != ".csv":
        raise ValueError("Choose a .csv file for the five-rule audit.")
    if len({path.resolve() for path, _ in frames}) != len(frames):
        raise ValueError("Choose distinct output paths for each table.")

    with ExitStack() as stack:
        workdirs = {}
        for directory in {path.parent for path, _ in frames}:
            directory.mkdir(parents=True, exist_ok=True)
            workdirs[directory] = Path(stack.enter_context(TemporaryDirectory(dir=directory)))
        outputs = {}
        staged = []
        for path, frame in frames:
            temp = workdirs[path.parent] / path.name
            if path.suffix == ".gz":
                with temp.open("wb") as stream:
                    with gzip.GzipFile(filename="", fileobj=stream, mode="wb", mtime=0, compresslevel=6) as zipped:
                        frame.to_csv(zipped, index=False, float_format="%.15g")
            else:
                frame.to_csv(temp, index=False)
            with temp.open("rb") as stream:
                digest = hashlib.file_digest(stream, "sha256").hexdigest()
            outputs[os.path.relpath(path, project)] = {"rows": len(frame), "sha256": digest}
            staged.append((temp, path))
        manifest_temp = workdirs[results] / manifest_path.name
        manifest_temp.write_text(json.dumps({**manifest, "outputs": outputs}, ensure_ascii=False, indent=2) + "\n",
                                 encoding="utf-8")
        for temp, path in staged:
            temp.replace(path)
        manifest_temp.replace(manifest_path)
manifest = cleaning_manifest(raw, baseline, steps, purchases, events)
save_cleaning_results(project, output, baseline, steps, decisions, purchases, events, registry, manifest)
print(json.dumps(manifest["counts"], indent=2))
print("Manifest / 文件清单：", project / "results/data_manifest.json")

{
  "after_five_rules": 392692,
  "removed_total": 150849,
  "paid_merchandise_lines": 391060,
  "basket_item_events": 386236,
  "baskets": 18402,
  "customers": 4334,
  "stock_codes": 3658,
  "paid_units": 5141694,
  "recorded_amount_gbp": 8735922.643
}
Manifest / 文件清单： /tmp/retail-final-cleaning-BYTFneR0/project/results/data_manifest.json
